In [20]:
# papermill parameters
RUN_DIR = "/work/runs/hybrids/hyb_v01_bm25_e5_k100"
RUN_CONTEXT = "manual"

In [21]:
# === metrics.ipynb — Per-method Evaluation Runner ===
#
# PURPOSE:
#   Load retrieval results from a single run (produced by retrieve.ipynb with SAVE=True)
#   and compute evaluation metrics at two levels:
#     1. Item-level (exact short→long match)
#     2. Parent-level (credit if query matches any sibling under the same parent)
#
# HOW IT WORKS:
#   1. Locate the latest results_top{K}.jsonl.gz in the chosen run directory (/work/runs/...).
#   2. Read index metadata to discover collection and parent structure.
#   3. Load processed queries (short items) and corpus (long items) from parquet.
#   4. For each query:
#        - Align with its gold document and parent.
#        - Compare ranked candidates from the run.
#        - Compute relevance vectors for item and parent targets.
#   5. Compute metrics:
#        • Acc@1, Recall@5, Recall@10
#        • MRR (Mean Reciprocal Rank)
#        • nDCG@10 (with proper parent normalization)
#   6. Split metrics by queries “with numbers” vs “without numbers.”
#   7. Save results to CSV and JSON:
#        - metrics_single__item.json/csv
#        - metrics_single__parent.json/csv
#        - metrics_dual.json/csv (both targets combined)
#
# KEY FEATURES:
#   • Dual-target evaluation (strict item vs. parent group).
#   • Automatic detection of parent column in the corpus.
#   • Numeric vs. non-numeric breakdown for error analysis.
#   • Standardized outputs for reproducible comparisons.
#
# Typical usage:
#   - Set RUN_DIR to the run you want to evaluate (e.g., /work/runs/bm25_unigram).
#   - Run all cells; metrics will be displayed and saved alongside the run folder.

# --- Single-method metrics: load ------------------------------------------------
import json, gzip, os, glob, math
import numpy as np
import pandas as pd
from pathlib import Path

# normalize params (must be in a separate, untagged cell)
RUN_DIR = Path(RUN_DIR)          # works whether RUN_DIR was str or Path
RUN_NAME = RUN_DIR.name

if not RUN_DIR.exists():
    raise FileNotFoundError(f"RUN_DIR does not exist: {RUN_DIR}")

print("[metrics] RUN_DIR:", RUN_DIR)


# find the results file written by retrieve.ipynb (results_top{K}.jsonl.gz)
res_files = sorted(Path(RUN_DIR).glob("results_top*.jsonl.gz"))
if not res_files:
    raise FileNotFoundError(f"No results_top*.jsonl.gz in {RUN_DIR}. "
                            "Run retrieve.ipynb with SAVE=True first.")
RESULTS_PATH = res_files[-1]  # pick the newest/biggest K

# peek first record to discover index_path (retrieve wrote this in meta)
first = None
with gzip.open(RESULTS_PATH, "rt", encoding="utf-8") as gz:
    for line in gz:
        if line.strip():
            first = json.loads(line)
            break
if first is None:
    raise RuntimeError(f"{RESULTS_PATH} is empty.")

index_path = Path(first["meta"]["index_path"])
index_meta = json.load(open(index_path / "meta.json", "r", encoding="utf-8"))
collection = index_meta.get("collection", "OEB")

# infer processed data root from index path (/work/index/.. -> /work/data/processed/)
repo_root = index_path.parent.parent    # /work
processed = repo_root / "data" / "processed"
QUERIES_PATH = processed / f"{collection}_short_feats.parquet"
CORPUS_PATH  = processed / f"{collection}_long_feats.parquet"

# load ground-truth tables
if not QUERIES_PATH.exists() or not CORPUS_PATH.exists():
    raise FileNotFoundError(f"Missing processed parquet files:\n  {QUERIES_PATH}\n  {CORPUS_PATH}\n")

df_q = pd.read_parquet(QUERIES_PATH)
df_d = pd.read_parquet(CORPUS_PATH)

# standardize text columns
df_q = df_q.rename(columns={"text_word":"q_text"})
df_d = df_d.rename(columns={"text_word":"d_text"})

# --- auto-detect parent column on the corpus (long docs) ---
PARENT_CANDIDATES = ["parent_key", "parent_id", "parent", "parent_item_key", "ParentID", "padre", "id_padre"]
PARENT_COL = next((c for c in PARENT_CANDIDATES if c in df_d.columns), None)
if PARENT_COL is None:
    raise KeyError(
        f"Could not find a parent column in corpus. Looked for {PARENT_CANDIDATES}. "
        f"Add/rename the parent column in {CORPUS_PATH}."
    )

# keep only needed columns to reduce RAM
df_q = df_q[["item_key", "q_text"]]
df_d = df_d[["item_key", "d_text", PARENT_COL]]

print("Loaded:")
print(" - Run dir      :", RUN_DIR)
print(" - Results file :", RESULTS_PATH.name)
print(" - Index path   :", index_path)
print(" - Collection   :", collection)
print(" - Parent col   :", PARENT_COL)
print(" - #queries     :", len(df_q), "| #docs:", len(df_d))



[metrics] RUN_DIR: /work/runs/hybrids/hyb_v01_bm25_e5_k100
Loaded:
 - Run dir      : /work/runs/hybrids/hyb_v01_bm25_e5_k100
 - Results file : results_top100.jsonl.gz
 - Index path   : /work/index/bm25_unigram
 - Collection   : OEB
 - Parent col   : parent_key
 - #queries     : 47513 | #docs: 47513


In [22]:
# --- Dual-target metrics (item + parent) with proper parent nDCG normalization ---
import math, gzip, json
import numpy as np
import pandas as pd
from IPython.display import display
from collections import defaultdict

# Lookups from Cell 1
gold_doc_keys = set(df_d["item_key"].astype(str))
q_table  = df_q.set_index(df_q["item_key"].astype(str), drop=False)
d_table  = df_d.set_index(df_d["item_key"].astype(str), drop=False)
doc2parent = d_table[PARENT_COL].astype(str).to_dict()

# Precompute parent -> set of doc keys (fast ideal DCG computation)
parent2docs = defaultdict(set)
for doc_k, parent_k in doc2parent.items():
    parent2docs[parent_k].add(doc_k)

def iter_results(jsonl_gz_path):
    with gzip.open(jsonl_gz_path, "rt", encoding="utf-8") as gz:
        for line in gz:
            if line.strip():
                yield json.loads(line)

def gold_label_for_query(qkey: str, target: str) -> str:
    return qkey if target == "item" else doc2parent.get(qkey, "")

def relevance_vector(cand_doc_keys, gold_label: str, target: str):
    if target == "item":
        return [1 if c == gold_label else 0 for c in cand_doc_keys]
    # parent
    rel_parent_docs = parent2docs.get(gold_label, set())
    return [1 if c in rel_parent_docs else 0 for c in cand_doc_keys]

def dcg_at_k(hh, k):
    s = 0.0
    for i, rel in enumerate(hh[:k], start=1):
        if rel:
            s += 1.0 / math.log2(i + 1)
    return s

def idcg_at_10_for_target(gold_label: str, target: str):
    """Ideal DCG@10: item → one relevant; parent → as many siblings as exist (capped at 10)."""
    if target == "item":
        # only one relevant item
        return dcg_at_k([1], 10)
    # parent target: all siblings are relevant
    num_rel = len(parent2docs.get(gold_label, ()))
    if num_rel == 0:
        return 0.0
    ideal = [1] * min(num_rel, 10)
    return dcg_at_k(ideal, 10)

def compute_metrics_for_target(target: str):
    """Returns (df_per_query, agg_series, K_seen)."""
    recs = []
    K_seen = None

    for rec in iter_results(RESULTS_PATH):
        qkey = str(rec.get("query_item_key", ""))
        if not qkey or qkey not in gold_doc_keys:
            continue
        cands = rec.get("candidates", [])
        if not cands:
            continue

        cand_doc_keys = [str(c.get("index_item_key","")) for c in cands]
        if K_seen is None:
            K_seen = len(cand_doc_keys)

        gold_lbl = gold_label_for_query(qkey, target)
        rel = relevance_vector(cand_doc_keys, gold_lbl, target)

        # Basic metrics
        acc1  = int(rel[0] == 1)
        rec5  = int(any(rel[:min(5, len(rel))]))
        rec10 = int(any(rel[:min(10, len(rel))]))

        # Reciprocal Rank
        rr = 0.0
        for r, v in enumerate(rel, start=1):
            if v:
                rr = 1.0 / r
                break

        # nDCG@10 with proper normalization
        dcg10  = dcg_at_k(rel, 10)
        idcg10 = idcg_at_10_for_target(gold_lbl, target)
        ndcg10 = (dcg10 / idcg10) if idcg10 > 0 else 0.0

        qtext = q_table.loc[qkey, "q_text"] if qkey in q_table.index else ""
        has_num = any(ch.isdigit() for ch in str(qtext))

        recs.append({
            "query_item_key": qkey,
            "acc1": acc1, "rec5": rec5, "rec10": rec10,
            "rr": rr, "ndcg10": ndcg10,
            "has_numbers": has_num
        })

    if not recs:
        raise RuntimeError(f"[{target}] No per-query records; check item_key/parents in parquet & results.")

    df = pd.DataFrame(recs)

    def agg(d):
        return pd.Series({
            "queries": len(d),
            "Acc@1":    float(d["acc1"].mean()),
            "Recall@5": float(d["rec5"].mean()),
            "Recall@10":float(d["rec10"].mean()),
            "MRR":      float(d["rr"].mean()),
            "nDCG@10":  float(d["ndcg10"].mean()),
        })

    overall = agg(df)
    split_has = agg(df[df["has_numbers"] == True])  if (df["has_numbers"] == True).any() else None
    split_no  = agg(df[df["has_numbers"] == False]) if (df["has_numbers"] == False).any() else None

    return df, overall, split_has, split_no, K_seen

# Run both targets
results = {}
combined_rows = []

for target in ["item", "parent"]:
    df_perq, overall, split_has, split_no, K_seen = compute_metrics_for_target(target)
    results[target] = {"df": df_perq, "overall": overall, "split_has": split_has, "split_no": split_no, "K": K_seen}

    # Display
    print(f"\nMethod: {RUN_DIR.name} | K in file: {K_seen} | TARGET: {target}"
          + (f" (parent col: {PARENT_COL})" if target == "parent" else ""))
    display(pd.DataFrame([overall]).round(4).assign(scope="Overall", target=target)[
        ["target","scope","queries","Acc@1","Recall@5","Recall@10","MRR","nDCG@10"]
    ])
    rows = []
    if split_has is not None:
        rows.append(pd.DataFrame([split_has]).assign(scope="Has numbers", target=target))
    if split_no is not None:
        rows.append(pd.DataFrame([split_no]).assign(scope="No numbers", target=target))
    if rows:
        display(pd.concat(rows, ignore_index=True).round(4)[
            ["target","scope","queries","Acc@1","Recall@5","Recall@10","MRR","nDCG@10"]
        ])

    # Save per-target
    suffix = "parent" if target == "parent" else "item"
    out_json = RUN_DIR / f"metrics_single__{suffix}.json"
    out_csv  = RUN_DIR / f"metrics_single__{suffix}.csv"
    report_rows = [{"method": RUN_DIR.name, **overall.to_dict(), "scope": "overall", "target": target}]
    if split_has is not None:
        report_rows.append({"method": RUN_DIR.name, **split_has.to_dict(), "scope": "has_numbers", "target": target})
    if split_no is not None:
        report_rows.append({"method": RUN_DIR.name, **split_no.to_dict(), "scope": "no_numbers", "target": target})
    df_report = pd.DataFrame(report_rows)
    df_report.to_json(out_json, orient="records", force_ascii=False, indent=2)
    df_report.to_csv(out_csv, index=False)
    print("Saved per-target:")
    print(" -", out_json)
    print(" -", out_csv)

    combined_rows.extend(df_report.to_dict(orient="records"))

# Save combined report (both targets side by side)
df_combined = pd.DataFrame(combined_rows)
df_combined.to_json(RUN_DIR / "metrics_dual.json", orient="records", force_ascii=False, indent=2)
df_combined.to_csv(RUN_DIR / "metrics_dual.csv", index=False)
print("\nSaved combined:")
print(" -", RUN_DIR / "metrics_dual.json")
print(" -", RUN_DIR / "metrics_dual.csv")


Method: hyb_v01_bm25_e5_k100 | K in file: 100 | TARGET: item


,target,scope,queries,Acc@1,Recall@5,Recall@10,MRR,nDCG@10
0,item,Overall,16590.0,0.341,0.8094,0.9237,0.55,0.6381


,target,scope,queries,Acc@1,Recall@5,Recall@10,MRR,nDCG@10
0,item,Has numbers,16556.0,0.3397,0.809,0.9235,0.5492,0.6374
1,item,No numbers,34.0,0.9706,1.000,1.0000,0.9853,0.9891


Saved per-target:
 - /work/runs/hybrids/hyb_v01_bm25_e5_k100/metrics_single__item.json
 - /work/runs/hybrids/hyb_v01_bm25_e5_k100/metrics_single__item.csv

Method: hyb_v01_bm25_e5_k100 | K in file: 100 | TARGET: parent (parent col: parent_key)


,target,scope,queries,Acc@1,Recall@5,Recall@10,MRR,nDCG@10
0,parent,Overall,16590.0,0.991,0.9952,0.9957,0.9931,0.9735


,target,scope,queries,Acc@1,Recall@5,Recall@10,MRR,nDCG@10
0,parent,Has numbers,16556.0,0.9909,0.9952,0.9957,0.9931,0.9734
1,parent,No numbers,34.0,1.0000,1.0000,1.0000,1.0000,1.0000


Saved per-target:
 - /work/runs/hybrids/hyb_v01_bm25_e5_k100/metrics_single__parent.json
 - /work/runs/hybrids/hyb_v01_bm25_e5_k100/metrics_single__parent.csv

Saved combined:
 - /work/runs/hybrids/hyb_v01_bm25_e5_k100/metrics_dual.json
 - /work/runs/hybrids/hyb_v01_bm25_e5_k100/metrics_dual.csv


In [23]:
# --- Explode results into a candidate-level table with correctness flags -------
import gzip, json
import pandas as pd

# Reuse variables from previous cells:
# RESULTS_PATH, RUN_DIR, df_q, df_d, PARENT_COL, d_table (index by item_key)

# Maps & helpers
doc2parent = d_table[PARENT_COL].astype(str).to_dict()
q_text_map = df_q.set_index("item_key")["q_text"].astype(str).to_dict()

rows = []
with gzip.open(RESULTS_PATH, "rt", encoding="utf-8") as gz:
    for line in gz:
        if not line.strip():
            continue
        rec = json.loads(line)
        qkey = str(rec.get("query_item_key", ""))
        qtext = q_text_map.get(qkey, "")
        gold_parent = doc2parent.get(qkey, "")

        for c in rec.get("candidates", []):
            rank = int(c.get("rank", 0))
            cand_key = str(c.get("index_item_key", ""))
            cand_parent = doc2parent.get(cand_key, "")
            rows.append({
                "query_id": rec.get("query_id",""),
                "query_item_key": qkey,
                "query_text": qtext,
                "gold_item_key": qkey,
                "gold_parent_key": gold_parent,
                "rank": rank,
                "cand_item_key": cand_key,
                "cand_parent_key": cand_parent,
                "score": float(c.get("score", 0.0)),
                "is_correct_item": int(cand_key == qkey),
                "is_correct_parent": int(cand_parent == gold_parent),
            })

df_cands = pd.DataFrame(rows)
print("Exploded candidates:", len(df_cands), "rows")
display(df_cands.head(10))

# Save for external inspection
cand_csv = RUN_DIR / "results_exploded_candidates.csv"
cand_parq = RUN_DIR / "results_exploded_candidates.parquet"
df_cands.to_csv(cand_csv, index=False)
df_cands.to_parquet(cand_parq, index=False)
print("Saved:", cand_csv, "\nSaved:", cand_parq)


Exploded candidates: 1659000 rows


,query_id,query_item_key,query_text,gold_item_key,gold_parent_key,rank,cand_item_key,cand_parent_key,score,is_correct_item,is_correct_parent
0,q_000000,OEB010aac,ejecución de canalización para línea subterran...,OEB010aac,OEB010$,1,OEB010aac,OEB010$,1.511789,1,1
1,q_000000,OEB010aac,ejecución de canalización para línea subterran...,OEB010aac,OEB010$,2,OEB010abc,OEB010$,1.486072,0,1
2,q_000000,OEB010aac,ejecución de canalización para línea subterran...,OEB010aac,OEB010$,3,OEB010aab,OEB010$,1.411923,0,1
3,q_000000,OEB010aac,ejecución de canalización para línea subterran...,OEB010aac,OEB010$,4,OEB010abb,OEB010$,1.404995,0,1
4,q_000000,OEB010aac,ejecución de canalización para línea subterran...,OEB010aac,OEB010$,5,OEB010aaa,OEB010$,1.403610,0,1
5,q_000000,OEB010aac,ejecución de canalización para línea subterran...,OEB010aac,OEB010$,6,OEB010aba,OEB010$,1.387469,0,1
6,q_000000,OEB010aac,ejecución de canalización para línea subterran...,OEB010aac,OEB010$,7,OEB050aac,OEB050$,1.369695,0,0
7,q_000000,OEB010aac,ejecución de canalización para línea subterran...,OEB010aac,OEB010$,8,OEB050abc,OEB050$,1.291639,0,0
8,q_000000,OEB010aac,ejecución de canalización para línea subterran...,OEB010aac,OEB010$,9,OEB010bac,OEB010$,1.139237,0,1
9,q_000000,OEB010aac,ejecución de canalización para línea subterran...,OEB010aac,OEB010$,10,OEB010cac,OEB010$,1.120937,0,1


Saved: /work/runs/hybrids/hyb_v01_bm25_e5_k100/results_exploded_candidates.csv 
Saved: /work/runs/hybrids/hyb_v01_bm25_e5_k100/results_exploded_candidates.parquet


In [24]:
# --- Per-query summary: first-hit ranks for item & parent ----------------------
import numpy as np

# first rank where item is correct (or np.inf)
first_item = (df_cands[df_cands["is_correct_item"]==1]
              .groupby("query_item_key")["rank"].min())
first_parent = (df_cands[df_cands["is_correct_parent"]==1]
                .groupby("query_item_key")["rank"].min())

summary = (
    df_cands[["query_item_key","query_text","gold_item_key","gold_parent_key"]]
    .drop_duplicates()
    .assign(rank_item=lambda d: d["query_item_key"].map(first_item).fillna(np.inf),
            rank_parent=lambda d: d["query_item_key"].map(first_parent).fillna(np.inf))
    .assign(
        acc1_item=lambda d: (d["rank_item"]==1).astype(int),
        acc1_parent=lambda d: (d["rank_parent"]==1).astype(int),
        rec5_item=lambda d: (d["rank_item"]<=5).astype(int),
        rec10_item=lambda d: (d["rank_item"]<=10).astype(int),
        rec5_parent=lambda d: (d["rank_parent"]<=5).astype(int),
        rec10_parent=lambda d: (d["rank_parent"]<=10).astype(int),
    )
    .sort_values(["acc1_parent","rank_parent","rank_item"], ascending=[True,True,True])
)

display(summary.head(10))

# Save
sum_csv = RUN_DIR / "results_perquery_summary.csv"
summary.to_csv(sum_csv, index=False)
print("Saved:", sum_csv)

# --- Find queries where gold item not in top-K -----------------
K = 100
missed = summary[summary["rank_item"] > K].copy()

missed_out = missed[["query_item_key", "query_text", "rank_item"]]
print("Missed queries:", len(missed_out))

# Save for inspection
missed_csv = RUN_DIR / "missed_queries.csv"
missed_out.to_csv(missed_csv, index=False)
print("Saved:", missed_csv)

# Peek a few
display(missed_out.head(10))


,query_item_key,query_text,gold_item_key,gold_parent_key,rank_item,rank_parent,acc1_item,acc1_parent,rec5_item,rec10_item,rec5_parent,rec10_parent
629000,OEB130abb,entronque de tubos de 110 mm en arqueta o cáma...,OEB130abb,OEB130$,2.0,2.0,0,0,1,1,1,1
629800,OEB130cac,entronque de tubos de 110 mm en arqueta o cáma...,OEB130cac,OEB130$,2.0,2.0,0,0,1,1,1,1
629900,OEB130cba,entronque de tubos de 110 mm en arqueta o cáma...,OEB130cba,OEB130$,2.0,2.0,0,0,1,1,1,1
630000,OEB130cbc,entronque de tubos de 110 mm en arqueta o cáma...,OEB130cbc,OEB130$,2.0,2.0,0,0,1,1,1,1
762800,OEB200bbaaa,zanja para cables de 0.80 a 1.00 m de profundi...,OEB200bbaaa,OEB200$,2.0,2.0,0,0,1,1,1,1
763400,OEB200bbbaa,zanja para cables de 0.80 a 1.00 m de profundi...,OEB200bbbaa,OEB200$,2.0,2.0,0,0,1,1,1,1
763800,OEB200bbcab,zanja para cables de 0.80 a 1.00 m de profundi...,OEB200bbcab,OEB200$,2.0,2.0,0,0,1,1,1,1
764200,OEB200bbdab,zanja para cables de 0.80 a 1.00 m de profundi...,OEB200bbdab,OEB200$,2.0,2.0,0,0,1,1,1,1
764300,OEB200bbdac,zanja para cables de 0.80 a 1.00 m de profundi...,OEB200bbdac,OEB200$,2.0,2.0,0,0,1,1,1,1
764700,OEB200bbdda,zanja para cables de 0.80 a 1.00 m de profundi...,OEB200bbdda,OEB200$,2.0,2.0,0,0,1,1,1,1


Saved: /work/runs/hybrids/hyb_v01_bm25_e5_k100/results_perquery_summary.csv
Missed queries: 147
Saved: /work/runs/hybrids/hyb_v01_bm25_e5_k100/missed_queries.csv


,query_item_key,query_text,rank_item
768400,OEB200cbabb,zanja para cables de 1.10 m de profundidad a m...,inf
768600,OEB200cbacb,zanja para cables de 1.10 m de profundidad a m...,inf
768700,OEB200cbacc,zanja para cables de 1.10 m de profundidad a m...,inf
769100,OEB200cbbcb,zanja para cables de 1.10 m de profundidad a m...,inf
769200,OEB200cbbcc,zanja para cables de 1.10 m de profundidad a m...,inf
769300,OEB200cbcbb,zanja para cables de 1.10 m de profundidad a m...,inf
756300,OEB200aacdc,zanja para cables de hasta 0.80 m de profundid...,inf
767500,OEB200caddc,zanja para cables de 1.10 m de profundidad a m...,inf
755300,OEB200aaadc,zanja para cables de hasta 0.80 m de profundid...,inf
757200,OEB200aaedc,zanja para cables de hasta 0.80 m de profundid...,inf


In [25]:
if RUN_CONTEXT != "manual":
    # skip during orchestrated runs
    raise SystemExit  # or just: pass

In [26]:
# --- Interactive inspector: show top-10 with item/parent flags -----------------
from IPython.display import display

def inspect(query_key: str, k: int = 10):
    s = summary.loc[summary["query_item_key"]==query_key]
    if s.empty:
        print("Query not found in summary:", query_key); return
    srow = s.iloc[0]
    print(f"\nQuery: {query_key}")
    print("Short text:", srow["query_text"][:240].replace("\n"," "))
    print("Gold item :", srow["gold_item_key"])
    print("Gold parent:", srow["gold_parent_key"])
    print(f"First correct item rank  : {int(srow['rank_item']) if srow['rank_item']!=np.inf else 'None'}")
    print(f"First correct parent rank: {int(srow['rank_parent']) if srow['rank_parent']!=np.inf else 'None'}")

    topk = (df_cands[df_cands["query_item_key"]==query_key]
            .sort_values("rank").head(k)
            [["rank","cand_item_key","cand_parent_key","score","is_correct_item","is_correct_parent"]])
    display(topk)

# Example calls:
# inspect("OEB040idbcb", k=10)
# inspect(summary.sort_values("rank_parent", ascending=False).iloc[0]["query_item_key"])


In [27]:
inspect("OEB040idbcb", k=10)

Query not found in summary: OEB040idbcb


In [28]:
inspect(summary.sort_values("rank_parent", ascending=False).iloc[0]["query_item_key"])


Query: OEB200caaca
Short text: zanja para cables de 1.10 m de profundidad a máquina normal en material normal diurno i 3 horas volumen relevante
Gold item : OEB200caaca
Gold parent: OEB200$
First correct item rank  : None
First correct parent rank: None


,rank,cand_item_key,cand_parent_key,score,is_correct_item,is_correct_parent
765900,1,OEB190cbcba,OEB190$,1.000000,0,0
765901,2,OEB190ccaca,OEB190$,1.000000,0,0
765902,3,OEB190ccaba,OEB190$,0.987366,0,0
765903,4,OEB190cccca,OEB190$,0.987366,0,0
765904,5,OEB190ccaaa,OEB190$,0.976992,0,0
765905,6,OEB190cccba,OEB190$,0.974777,0,0
765906,7,OEB190ccada,OEB190$,0.966314,0,0
765907,8,OEB190cccaa,OEB190$,0.964400,0,0
765908,9,OEB190cccda,OEB190$,0.953743,0,0
765909,10,OEB190bbcba,OEB190$,0.928176,0,0


In [29]:
# --- Find cases where parent Acc@1 is 0 but parent is in top-5/10 -------------
bad_parent_at1 = summary[summary["acc1_parent"]==0].copy()
print("Queries with parent@1 = 0:", len(bad_parent_at1))

# Parent present in top-5/10?
in5 = bad_parent_at1[bad_parent_at1["rec5_parent"]==1]
in10 = bad_parent_at1[bad_parent_at1["rec10_parent"]==1]
print("... of which parent in top-5:", len(in5))
print("... of which parent in top-10:", len(in10))

# Peek a few
peek = in5.sample(min(5, len(in5)), random_state=0) if len(in5)>0 else pd.DataFrame()
for _, r in peek.iterrows():
    inspect(r["query_item_key"], k=10)


Queries with parent@1 = 0: 150
... of which parent in top-5: 71
... of which parent in top-10: 78

Query: OEB230ihdcb
Short text: canalización hormigonada 16 t polietileno libre de halógenos de 200 mm con topo nocturno excepcional i 3 horas volumen escaso
Gold item : OEB230ihdcb
Gold parent: OEB230$
First correct item rank  : 2
First correct parent rank: 2


,rank,cand_item_key,cand_parent_key,score,is_correct_item,is_correct_parent
953100,1,OEB040bddcb,OEB040$,1.000000,0,0
953101,2,OEB230ihdcb,OEB230$,1.000000,1,1
953102,3,OEB230ihdbb,OEB230$,0.992026,0,1
953103,4,OEB040bdddb,OEB040$,0.927486,0,0
953104,5,OEB230ihdab,OEB230$,0.920172,0,1
953105,6,OEB230ihbcb,OEB230$,0.902039,0,1
953106,7,OEB230iddbb,OEB230$,0.896064,0,1
953107,8,OEB230iddcb,OEB230$,0.894959,0,1
953108,9,OEB230ihbbb,OEB230$,0.894223,0,1
953109,10,OEB230ihddb,OEB230$,0.883129,0,1



Query: OEB230ihddb
Short text: canalización hormigonada 16 t polietileno libre de halógenos de 200 mm con topo nocturno excepcional no necesita intervalo volumen escaso
Gold item : OEB230ihddb
Gold parent: OEB230$
First correct item rank  : 2
First correct parent rank: 2


,rank,cand_item_key,cand_parent_key,score,is_correct_item,is_correct_parent
953300,1,OEB040bdddb,OEB040$,1.000000,0,0
953301,2,OEB230ihddb,OEB230$,1.000000,1,1
953302,3,OEB230iadda,OEB230$,0.986472,0,1
953303,4,OEB230idddb,OEB230$,0.950668,0,1
953304,5,OEB230ihbdb,OEB230$,0.933293,0,1
953305,6,OEB230iaddb,OEB230$,0.873791,0,1
953306,7,OEB230iddda,OEB230$,0.869716,0,1
953307,8,OEB040badda,OEB040$,0.855400,0,0
953308,9,OEB040baddb,OEB040$,0.849696,0,0
953309,10,OEB040bddda,OEB040$,0.840677,0,0



Query: OEB200abbcc
Short text: zanja para cables de hasta 0.80 m de profundidad a máquina rocoso en material rocoso nocturno i 3 horas cualquier condición de ejecución
Gold item : OEB200abbcc
Gold parent: OEB200$
First correct item rank  : 98
First correct parent rank: 2


,rank,cand_item_key,cand_parent_key,score,is_correct_item,is_correct_parent
758300,1,OEB190acbcc,OEB190$,1.000000,0,0
758301,2,OEB200abdda,OEB200$,1.000000,0,1
758302,3,OEB200abddc,OEB200$,0.985569,0,1
758303,4,OEB190acbbc,OEB190$,0.981909,0,0
758304,5,OEB190acdcc,OEB190$,0.981909,0,0
758305,6,OEB190acbac,OEB190$,0.972562,0,0
758306,7,OEB190acdbc,OEB190$,0.963878,0,0
758307,8,OEB190acbdc,OEB190$,0.959829,0,0
758308,9,OEB190acdac,OEB190$,0.954519,0,0
758309,10,OEB190acddc,OEB190$,0.941810,0,0



Query: OEB230ihcca
Short text: canalización hormigonada 16 t polietileno libre de halógenos de 200 mm con topo diurno excepcional i 3 horas volumen relevante
Gold item : OEB230ihcca
Gold parent: OEB230$
First correct item rank  : 2
First correct parent rank: 2


,rank,cand_item_key,cand_parent_key,score,is_correct_item,is_correct_parent
952700,1,OEB040bdcda,OEB040$,1.000000,0,0
952701,2,OEB230ihcca,OEB230$,1.000000,1,1
952702,3,OEB230ihcba,OEB230$,0.992026,0,1
952703,4,OEB040bdcdb,OEB040$,0.963545,0,0
952704,5,OEB230ihcaa,OEB230$,0.920172,0,1
952705,6,OEB230ihaca,OEB230$,0.902039,0,1
952706,7,OEB230ihaba,OEB230$,0.894223,0,1
952707,8,OEB230ihcda,OEB230$,0.883129,0,1
952708,9,OEB230ihaaa,OEB230$,0.822072,0,1
952709,10,OEB040bdcca,OEB040$,0.821321,0,0



Query: OEB230ihebc
Short text: canalización hormigonada 16 t polietileno libre de halógenos de 200 mm con topo cualquier franja horaria 3 i 5 horas cualquier condición de ejecución
Gold item : OEB230ihebc
Gold parent: OEB230$
First correct item rank  : 2
First correct parent rank: 2


,rank,cand_item_key,cand_parent_key,score,is_correct_item,is_correct_parent
953600,1,OEB040effdc,OEB040$,1.000000,0,0
953601,2,OEB230ihebc,OEB230$,1.000000,1,1
953602,3,OEB230ihfbc,OEB230$,0.991653,0,1
953603,4,OEB230iheac,OEB230$,0.928659,0,1
953604,5,OEB230ihecc,OEB230$,0.925117,0,1
953605,6,OEB230ihfac,OEB230$,0.920422,0,1
953606,7,OEB230ihfcc,OEB230$,0.916886,0,1
953607,8,OEB040fffdc,OEB040$,0.905552,0,0
953608,9,OEB230ihedc,OEB230$,0.808410,0,1
953609,10,OEB230ihfdc,OEB230$,0.800381,0,1


In [30]:
# --- Confusion analysis: most frequent wrong parent@1 --------------------------
rank1 = df_cands[df_cands["rank"]==1].copy()
rank1["is_parent_hit"] = (rank1["cand_parent_key"]==rank1["gold_parent_key"]).astype(int)
wrong_parent = rank1[rank1["is_parent_hit"]==0]

tbl = (wrong_parent
       .groupby(["gold_parent_key","cand_parent_key"])
       .size().reset_index(name="count")
       .sort_values("count", ascending=False)
       .head(20))
display(tbl)

# Save
tbl.to_csv(RUN_DIR / "parent_confusions_rank1.csv", index=False)
print("Saved:", RUN_DIR / "parent_confusions_rank1.csv")


,gold_parent_key,cand_parent_key,count
2,OEB200$,OEB190$,114
3,OEB230$,OEB040$,23
1,OEB130$,OEB120$,6
0,OEB050$,OEB010$,5
4,OEB280$,OEB040$,1
5,OEB290$,OEB040$,1


Saved: /work/runs/hybrids/hyb_v01_bm25_e5_k100/parent_confusions_rank1.csv


In [31]:
if RUN_CONTEXT != "manual":
    # skip during orchestrated runs
    raise SystemExit  # or just: pass

In [32]:
# look at the long-format item OEB060
doc_text = df_d.loc[df_d["item_key"]=="OEB060", "d_text"].iloc[0]
print("Length (chars):", len(doc_text))
print("Length (tokens):", len(doc_text.split()))
print("Text:\n", doc_text)



Length (chars): 279
Length (tokens): 53
Text:
 canalización de 5 tubos de acero de 3 de diámetro en placa de hormigón de 30 cm de profundidad mínima incluida la acometida y el remate a la canaleta o a la arqueta existente y la reposición de la placa de hormigón también se incluyen los accesorios y las uniones entre los tubos


In [33]:
df_d.loc[df_d["item_key"]=="OEB060"].T


,17316
item_key,OEB060
d_text,canalización de 5 tubos de acero de 3 de diáme...
parent_key,OEB060
